In [0]:
%pip install pymupdf

In [0]:
dbutils.library.restartPython()

### For PDF (discharge_summaries)

In [0]:
# Read PDF files from the Volume
from pyspark.sql import functions as F

PDF_PATH = "/Volumes/claims/raw/landing/genai/discharge_summaries/"

pdf_df = (
    spark.read
    .format("binaryFile")
    .option("pathGlobFilter", "*.pdf")
    .load(PDF_PATH)
    .select(
        F.col("path"),
        F.col("content"),
        F.element_at(
            F.split(F.col("path"), "/"), -1
        ).alias("source_file")
    )
)

print("Total PDF files:", pdf_df.count())

display(pdf_df.select("path", "source_file"))

In [0]:
# Parse the PDF using AI Functions
parsed_df = (
    pdf_df
    .withColumn(
        "parsed_document",
        F.expr("""
            ai_parse_document(
                content,
                map('version', '2.0')
            )
        """)
    )
)

display(
    parsed_df.select(
        "source_file",
        "parsed_document"
    )
)

In [0]:
# Extract the complete document text
parsed_text_df = parsed_df.withColumn(
    "document_text",
    F.expr("""
        concat_ws(
            '\\n',
            transform(
                from_json(
                    to_json(parsed_document:document:elements),
                    'array<struct<content:string>>'
                ),
                x -> x.content
            )
        )
    """)
)

display(
    parsed_text_df.select(
        "source_file",
        "document_text"
    )
)

In [0]:
# Define the AI extraction schema
extraction_schema = {
    "claim_id": {
        "type": "string",
        "description": "Claim reference from the discharge summary, such as CLM-10072."
    },
    "diagnosis_code": {
        "type": "string",
        "description": "ICD-10 code of the FINAL diagnosis only. Ignore older, historical, unrelated, or provisional diagnoses."
    },
    "procedure_code": {
        "type": "string",
        "description": "Procedure code corresponding to the procedure performed during this hospital admission."
    },
    "admission_date": {
        "type": "string",
        "description": "Date of admission in YYYY-MM-DD format."
    },
    "discharge_date": {
        "type": "string",
        "description": "Date of discharge in YYYY-MM-DD format."
    },
    "room_type": {
        "type": "enum",
        "description": "Normalize the room category to one of: general_ward, twin_sharing, single_private, deluxe.",
        "values": [
            "general_ward",
            "twin_sharing",
            "single_private",
            "deluxe"
        ]
    }
}

In [0]:
# Run ai_extract()
from pyspark.sql.functions import ai_extract

# ai_extract expects 'labels' (not 'values') for enum-type fields
for field_schema in extraction_schema.values():
    if "values" in field_schema:
        field_schema["labels"] = field_schema.pop("values")

extracted_df = (
    parsed_text_df
    .withColumn(
        "ai_result",
        ai_extract(
            F.col("parsed_document"),
            extraction_schema,
            {
                "version": "2.1",
                "instructions": """
                    You are extracting structured information
                    from Saffron Shield hospital discharge summaries.

                    Extract the claim reference and five required facts.

                    IMPORTANT RULES:

                    1. Extract the ICD-10 code of the FINAL diagnosis.
                    2. Ignore older, unrelated, historical, or provisional diagnoses.
                    3. Extract the procedure code of the procedure actually performed.
                    4. Extract admission and discharge dates accurately.
                    5. Normalize room types to the supplied enum values.
                    6. Do not invent missing information.
                    7. Return null if a field cannot be identified.
                """
            }
        )
    )
)

display(
    extracted_df.select(
        "source_file",
        "ai_result"
    )
)

In [0]:
# Convert AI results into the required columns
result_df = (
    extracted_df
    .withColumn("result_json", F.to_json("ai_result"))
    .select(
        F.get_json_object(
            "result_json", "$.response.claim_id.value"
        ).alias("claim_id"),

        "source_file",

        F.lit("pdf").alias("source"),

        "document_text",

        F.get_json_object(
            "result_json", "$.response.diagnosis_code.value"
        ).alias("diagnosis_code"),

        F.get_json_object(
            "result_json", "$.response.procedure_code.value"
        ).alias("procedure_code"),

        F.to_date(
            F.get_json_object(
                "result_json", "$.response.admission_date.value"
            )
        ).alias("admission_date"),

        F.to_date(
            F.get_json_object(
                "result_json", "$.response.discharge_date.value"
            )
        ).alias("discharge_date"),

        F.get_json_object(
            "result_json", "$.response.room_type.value"
        ).alias("room_type")
    )
)

display(result_df)

In [0]:
(
    result_df
    .select(
        "claim_id",
        "source_file",
        "source",
        "document_text",
        "diagnosis_code",
        "procedure_code",
        "admission_date",
        "discharge_date",
        "room_type"
    )
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("claims.genai.discharge_extracted")
)

In [0]:
%sql
select * from claims.genai.discharge_extracted

In [0]:
%sql

WITH comparison AS (
    SELECT
        e.claim_id,

        CASE
            WHEN UPPER(TRIM(e.diagnosis_code))
               = UPPER(TRIM(c.diagnosis_code))
            THEN 1 ELSE 0
        END AS diagnosis_match,

        CASE
            WHEN UPPER(TRIM(e.procedure_code))
               = UPPER(TRIM(c.procedure_code))
            THEN 1 ELSE 0
        END AS procedure_match,

        CASE
            WHEN e.admission_date = c.admission_date
            THEN 1 ELSE 0
        END AS admission_match,

        CASE
            WHEN e.discharge_date = c.discharge_date
            THEN 1 ELSE 0
        END AS discharge_match

    FROM claims.genai.discharge_extracted e

    INNER JOIN claims.silver.claims_clean c
        ON e.claim_id = c.claim_id
),

scored AS (
    SELECT
        *,
        diagnosis_match +
        procedure_match +
        admission_match +
        discharge_match AS matched_fields
    FROM comparison
)

SELECT
    COUNT(*) AS compared_claims,
    SUM(matched_fields) AS matching_fields,
    COUNT(*) * 4 AS total_fields,

    ROUND(
        100.0 * SUM(matched_fields)
        / NULLIF(COUNT(*) * 4, 0),
        2
    ) AS match_percentage

FROM scored;

### For Txt (discharge_summaries_txt)

In [0]:
# # Configure the input path
# from pyspark.sql import functions as F
# from pyspark.sql.types import *
# from datetime import datetime
# import re
# import os

# # Use TXT files so every output row has source = 'txt'
# SOURCE_FORMAT = "txt"

# INPUT_PATH = (
#     "/Volumes/claims/raw/landing/genai/"
#     "discharge_summaries_txt/"
# )

# TARGET_TABLE = "claims.genai.discharge_extracted"

In [0]:
# # Create extraction functions
# def extract_field(pattern, text):
#     match = re.search(
#         pattern,
#         text,
#         re.IGNORECASE | re.MULTILINE
#     )

#     return match.group(1).strip() if match else None


# def parse_date(date_value):
#     if not date_value:
#         return None

#     formats = [
#         "%b %d, %Y",       # Jan 20, 2026
#         "%B %d, %Y",       # January 20, 2026
#         "%d/%m/%Y",
#         "%d-%m-%Y",
#         "%Y-%m-%d",
#         "%d %B %Y"
#     ]

#     for fmt in formats:
#         try:
#             return datetime.strptime(date_value.strip(), fmt).date()
#         except ValueError:
#             continue

#     return None


# def extract_room_type(room_value):
#     if not room_value:
#         return None

#     value = room_value.lower()

#     if "general ward" in value:
#         return "general_ward"

#     if "twin sharing" in value:
#         return "twin_sharing"

#     if "single private" in value:
#         return "single_private"

#     if "deluxe" in value:
#         return "deluxe"

#     return None


# def extract_discharge_summary(document_text, source_file):

#     # Claim reference
#     claim_id = extract_field(
#         r"Claim\s+reference\s*:\s*([A-Z0-9-]+)",
#         document_text
#     )

#     # Final diagnosis ICD-10 code
#     diagnosis_code = extract_field(
#         r"Final\s+diagnosis\s*:.*?\(ICD-10\s+([A-Z0-9.]+)\)",
#         document_text
#     )

#     # Procedure code
#     procedure_code = extract_field(
#         r"Procedure\s+performed\s*:.*?\(procedure\s+code\s+([A-Z0-9-]+)\)",
#         document_text
#     )

#     # Admission date
#     admission_raw = extract_field(
#         r"Date\s+of\s+admission\s*:\s*([^\r\n]+)",
#         document_text
#     )

#     # Discharge date
#     discharge_raw = extract_field(
#         r"Date\s+of\s+discharge\s*:\s*([^\r\n]+)",
#         document_text
#     )

#     # Room category
#     room_raw = extract_field(
#         r"Room\s+category\s*:\s*([^\r\n(]+)",
#         document_text
#     )

#     return {
#         "claim_id": claim_id,
#         "source_file": source_file,
#         "source": SOURCE_FORMAT,
#         "document_text": document_text,
#         "diagnosis_code": diagnosis_code,
#         "procedure_code": procedure_code,
#         "admission_date": parse_date(admission_raw),
#         "discharge_date": parse_date(discharge_raw),
#         "room_type": extract_room_type(room_raw)
#     }

In [0]:
# # Read all TXT files from your Unity Catalog Volume
# files_df = (
#     spark.read
#     .format("binaryFile")
#     .option("pathGlobFilter", "*.txt")
#     .load(INPUT_PATH)
#     .select("path", "content")
# )

# extracted_rows = []

# for row in files_df.toLocalIterator():

#     try:
#         document_text = row["content"].decode(
#             "utf-8-sig",
#             errors="replace"
#         )

#         source_file = row["path"].rsplit("/", 1)[-1]

#         result = extract_discharge_summary(
#             document_text,
#             source_file
#         )

#         extracted_rows.append(result)

#     except Exception as e:
#         print(
#             f"Error processing {row['path']}: {str(e)}"
#         )

# print("Total documents extracted:", len(extracted_rows))

In [0]:
# # Create the DataFrame
# output_schema = StructType([
#     StructField("claim_id", StringType(), True),
#     StructField("source_file", StringType(), True),
#     StructField("source", StringType(), False),
#     StructField("document_text", StringType(), True),
#     StructField("diagnosis_code", StringType(), True),
#     StructField("procedure_code", StringType(), True),
#     StructField("admission_date", DateType(), True),
#     StructField("discharge_date", DateType(), True),
#     StructField("room_type", StringType(), True)
# ])

# extracted_df = spark.createDataFrame(
#     extracted_rows,
#     schema=output_schema
# )

# display(extracted_df)

In [0]:
# # Write to claims.genai.discharge_extracted
# (
#     extracted_df
#     .write
#     .format("delta")
#     .mode("append")
#     .option("overwriteSchema", "true")
#     .saveAsTable(TARGET_TABLE)
# )

In [0]:
%sql
CREATE OR REPLACE TABLE claims.genai.grievance_emails AS
SELECT *
FROM read_files(
    '/Volumes/claims/raw/landing/genai/grievance_emails.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

In [0]:
%sql
select * from claims.genai.grievance_emails

In [0]:
%sql
CREATE OR REPLACE TABLE claims.genai.grievance_classified AS

WITH classified AS (
    SELECT
        email_id,

        ai_classify(
            CONCAT(
                'Subject: ', COALESCE(subject, ''),
                '\nEmail body: ', COALESCE(body, '')
            ),

            '{
                "claim_delay": "The customer complains about claim processing, approval, or settlement taking too long, or a claim remaining pending without an update.",
                "claim_rejection": "The customer disputes a claim rejection, denial, or refusal by the insurer, including rejection due to waiting periods or policy conditions.",
                "payout_deduction": "The customer complains about deductions, reduced settlement amounts, short payments, or receiving less money than expected.",
                "cashless_network": "The customer complains about cashless hospital admission, network hospital availability, or cashless authorization issues.",
                "policy_service": "The customer complains about policy servicing, renewal, policy documents, corrections, member details, or general policy administration."
            }',

            MAP(
                'version', '2.1',
                'instructions',
                'Classify each email into exactly one category based on its primary complaint. Consider both the subject and body. Ignore secondary or incidental issues. If the customer disputes a rejection, choose claim_rejection. If the complaint is about a pending claim or delayed processing, choose claim_delay. If the complaint is about a reduced payout or deduction, choose payout_deduction. If it concerns cashless hospital access, choose cashless_network. If it concerns general policy administration, choose policy_service. Return the single best matching category.',
                'enableConfidenceScores', 'true',
                'enableRationales', 'true'
            )
        ) AS classification

    FROM claims.genai.grievance_emails
)

SELECT
    email_id,
    get_json_object(
        to_json(classification),
        '$.response[0].value'
    ) AS category
FROM classified;

### R2 - Chunk, Embed and Index

In [0]:

import re
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# ============================================================
# CONFIGURATION
# ============================================================

SOURCE_PATH = "/Volumes/claims/raw/landing/genai/policy_docs/"
TARGET_TABLE = "claims.genai.policy_chunks"

# ============================================================
# 1. READ ALL MARKDOWN POLICY DOCUMENTS
# ============================================================

files_df = (
    spark.read
    .format("binaryFile")
    .option("pathGlobFilter", "*.md")
    .load(SOURCE_PATH)
    .select("path", "content")
)

# Collect document contents to the driver.
# Appropriate for this small binder of eight documents.
documents = files_df.collect()

print(f"Total policy documents found: {len(documents)}")


# ============================================================
# 2. HELPER FUNCTION TO GENERATE STABLE CHUNK ID
# ============================================================

def generate_chunk_id(filename, heading):

    # Lowercase the heading
    anchor = heading.lower()

    # Replace every run of non-alphanumeric characters with "-"
    anchor = re.sub(r"[^a-z0-9]+", "-", anchor)

    # Remove leading and trailing "-"
    anchor = anchor.strip("-")

    # Filename keeps original case and contains no folder
    return f"{filename}#{anchor}"


# ============================================================
# 3. SPLIT DOCUMENT INTO SECTIONS
# ============================================================

def split_into_chunks(filename, document_text):

    # Normalize line endings
    document_text = document_text.replace("\r\n", "\n").replace("\r", "\n")

    # Find every Markdown ## heading
    heading_pattern = re.compile(r"(?m)^##[ \t]+(.+?)[ \t]*$")

    matches = list(heading_pattern.finditer(document_text))

    chunks = []

    # Document header = everything before first ##
    if not matches:
        print(f"WARNING: No ## sections found in {filename}")
        return chunks

    document_header = document_text[:matches[0].start()].strip()

    for index, match in enumerate(matches):

        heading = match.group(1).strip()

        # Section body extends until the next ## heading or EOF
        section_start = match.end()

        if index + 1 < len(matches):
            section_end = matches[index + 1].start()
        else:
            section_end = len(document_text)

        section_body = document_text[section_start:section_end].strip()

        # Construct exact chunk text:
        # document header
        # blank line
        # ## heading
        # blank line
        # section body

        heading_line = f"## {heading}"

        chunk_text = (
            f"{document_header}\n\n"
            f"{heading_line}\n\n"
            f"{section_body}"
        )

        # Filename without folder
        chunk_id = generate_chunk_id(filename, heading)

        chunks.append({
            "chunk_id": chunk_id,
            "doc_name": filename,
            "heading": heading,
            "chunk_index": index,
            "text": chunk_text
        })

    return chunks


# ============================================================
# 4. PROCESS ALL DOCUMENTS
# ============================================================

all_chunks = []

for document in documents:

    # Extract filename without folder
    filename = document["path"].rsplit("/", 1)[-1]

    # Decode PDF-independent Markdown text
    document_text = bytes(document["content"]).decode("utf-8")

    document_chunks = split_into_chunks(
        filename,
        document_text
    )

    all_chunks.extend(document_chunks)

    print(
        f"{filename}: {len(document_chunks)} sections extracted"
    )


print(f"\nTotal chunks generated: {len(all_chunks)}")


# ============================================================
# 5. CREATE SPARK DATAFRAME
# ============================================================

schema = """
    chunk_id STRING,
    doc_name STRING,
    heading STRING,
    chunk_index INT,
    text STRING
"""

chunks_df = spark.createDataFrame(
    all_chunks,
    schema=schema
)


# ============================================================
# 6. VALIDATE GENERATED CHUNKS BEFORE WRITING
# ============================================================

# Check duplicate chunk IDs
duplicate_ids = (
    chunks_df
    .groupBy("chunk_id")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_ids.limit(1).count() > 0:
    display(duplicate_ids)
    raise ValueError(
        "Duplicate chunk IDs found. Resolve duplicate headings "
        "within documents before loading the table."
    )


# Check required fields
invalid_chunks = chunks_df.filter(
    F.col("chunk_id").isNull()
    | F.col("doc_name").isNull()
    | F.col("heading").isNull()
    | F.col("text").isNull()
)

if invalid_chunks.limit(1).count() > 0:
    display(invalid_chunks)
    raise ValueError("Invalid chunks found. Aborting load.")

In [0]:
# ============================================================
# 7. CREATE TARGET TABLE IF IT DOES NOT EXIST
# ============================================================

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
        chunk_id STRING NOT NULL,
        doc_name STRING,
        heading STRING,
        chunk_index INT,
        text STRING
    )
    USING DELTA
""")


# Enable Change Data Feed
spark.sql(f"""
    ALTER TABLE {TARGET_TABLE}
    SET TBLPROPERTIES (
        delta.enableChangeDataFeed = true
    )
""")

In [0]:
# ============================================================
# 8. INCREMENTAL SYNCHRONIZATION
# ============================================================

if not spark.catalog.tableExists(TARGET_TABLE):

    raise RuntimeError("Target table creation failed.")

# Use the latest complete snapshot of the eight documents.
# Delete missing chunks, update changed chunks, insert new chunks.

delta_table = DeltaTable.forName(spark, TARGET_TABLE)

(
    delta_table.alias("target")
    .merge(
        chunks_df.alias("source"),
        "target.chunk_id = source.chunk_id"
    )
    .whenMatchedUpdate(
        condition="""
            NOT (
                target.doc_name <=> source.doc_name
                AND target.heading <=> source.heading
                AND target.chunk_index <=> source.chunk_index
                AND target.text <=> source.text
            )
        """,
        set={
            "doc_name": "source.doc_name",
            "heading": "source.heading",
            "chunk_index": "source.chunk_index",
            "text": "source.text"
        }
    )
    .whenNotMatchedInsert(
        values={
            "chunk_id": "source.chunk_id",
            "doc_name": "source.doc_name",
            "heading": "source.heading",
            "chunk_index": "source.chunk_index",
            "text": "source.text"
        }
    )
    .whenNotMatchedBySourceDelete()
    .execute()
)


print("Policy chunk table synchronized successfully.")

In [0]:
%sql
SELECT
    COUNT(*) AS total_chunks,
    COUNT(DISTINCT doc_name) AS total_documents,
    COUNT(DISTINCT chunk_id) AS unique_chunk_ids
FROM claims.genai.policy_chunks;

In [0]:
%sql
SELECT
    version,
    timestamp,
    operation,
    operationMetrics
FROM (
    DESCRIBE HISTORY claims.genai.policy_chunks
)
WHERE operation = 'MERGE'
ORDER BY version DESC;

In [0]:
%sql
SELECT
    COUNT(*) AS total_chunks,
    COUNT(DISTINCT chunk_id) AS unique_chunks,
    COUNT_IF(text IS NULL OR TRIM(text) = '') AS empty_chunks
FROM claims.genai.policy_chunks;

### R2 - Create the Delta Sync index

In [0]:
%pip install -U databricks-vectorsearch databricks-ai-search databricks-langchain langchain-openai langchain-core pydantic sentence-transformers mlflow

In [0]:
dbutils.library.restartPython()

In [0]:
# Create the endpoint
from databricks.ai_search.client import AISearchClient

client = AISearchClient()

client.create_endpoint(
    name="claims_vs",
    endpoint_type="STANDARD"
)

print("AI Search endpoint creation requested.")

In [0]:
# Create the index
index = client.create_delta_sync_index(
    endpoint_name="claims_vs",
    index_name="claims.genai.policy_chunks_idx",
    source_table_name="claims.genai.policy_chunks",
    pipeline_type="TRIGGERED",
    primary_key="chunk_id",
    embedding_source_column="text",
    embedding_model_endpoint_name="databricks-gte-large-en"
)

print("Delta Sync index creation requested.")

In [0]:
index = client.get_index(
    endpoint_name="claims_vs",
    index_name="claims.genai.policy_chunks_idx"
)

index.sync()

In [0]:
from databricks_langchain import ChatDatabricks
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

In [0]:
# Credentials come from the notebook session automatically — never hardcode tokens
from databricks.sdk import WorkspaceClient

_w = WorkspaceClient()
_auth_headers = _w.config.authenticate()

workspace_url = _w.config.host.rstrip("/")
workspace_token = _auth_headers["Authorization"].removeprefix("Bearer ")

LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

In [0]:
chat_databricks = ChatDatabricks(
    endpoint=LLM_ENDPOINT,
    temperature=0
)

response_db = chat_databricks.invoke([
    HumanMessage(
        content="Explain what a health insurance waiting period means in two sentences."
    )
])

chat_databricks_reply = response_db.content

print(chat_databricks_reply)

In [0]:
chat_openai = ChatOpenAI(
    model=LLM_ENDPOINT,
    base_url=f"{workspace_url}/serving-endpoints",
    api_key=workspace_token,
    temperature=0
)

response_openai = chat_openai.invoke([
    HumanMessage(
        content="Explain what a health insurance waiting period means in two sentences."
    )
])

chat_openai_reply = response_openai.content

print(chat_openai_reply)

In [0]:
# Define the typed Pydantic output schema
from pydantic import BaseModel, Field

class WaitingPeriodFacts(BaseModel):

    document_name: str = Field(
        description="Exact policy document filename"
    )

    plan_family: str = Field(
        description="Insurance plan family described in the document"
    )

    initial_waiting_days: int = Field(
        ge=0,
        description="Initial waiting period in days"
    )

    pre_existing_disease_waiting_months: int = Field(
        ge=0,
        description="Pre-existing disease waiting period in months"
    )

In [0]:
# Create the extraction chain
from langchain_core.prompts import ChatPromptTemplate

structured_llm = chat_databricks.with_structured_output(
    WaitingPeriodFacts,
    method="function_calling"
)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are an insurance policy document extraction assistant.

        Extract waiting-period information only from the supplied policy document.

        Return these fields:
        1. document_name
        2. plan_family
        3. initial_waiting_days
        4. pre_existing_disease_waiting_months

        Rules:
        - Use only information explicitly supported by the document.
        - Do not guess or use general insurance knowledge.
        - Extract the initial waiting period in days.
        - Extract the pre-existing disease waiting period in months.
        - If the document expresses a duration in years, convert it to months.
        - Do not confuse specific disease waiting periods with the
          pre-existing disease waiting period.
        - Preserve the exact document filename.
        """
    ),
    (
        "human",
        """
        Document filename:
        {document_name}

        Policy document content:
        {document_text}

        Extract the waiting-period facts.
        """
    )
])

extraction_chain = prompt | structured_llm

In [0]:
# finding documents begin with 02 and 03
import os
from pathlib import Path

POLICY_PATH = "/Volumes/claims/raw/landing/genai/policy_docs/"

files = os.listdir(POLICY_PATH)

doc_02 = next(
    (f for f in files if f.startswith("02") and f.endswith(".md")),
    None
)

doc_03 = next(
    (f for f in files if f.startswith("03") and f.endswith(".md")),
    None
)

if not doc_02 or not doc_03:
    raise FileNotFoundError(
        f"Could not find both waiting-period documents. "
        f"Found: {doc_02}, {doc_03}"
    )

print("Document 02:", doc_02)
print("Document 03:", doc_03)

In [0]:
# Read the document contents:
def read_policy_document(filename):

    file_path = os.path.join(POLICY_PATH, filename)

    with open(file_path, "r", encoding="utf-8") as file:
        return file.read()


document_02_text = read_policy_document(doc_02)
document_03_text = read_policy_document(doc_03)

print("Document 02 characters:", len(document_02_text))
print("Document 03 characters:", len(document_03_text))

In [0]:
# Extract structured facts from both documents
waiting_facts = []

for filename, document_text in [
    (doc_02, document_02_text),
    (doc_03, document_03_text)
]:

    extracted = extraction_chain.invoke({
        "document_name": filename,
        "document_text": document_text
    })

    # Ensure the exact source filename is retained.
    extracted = extracted.model_copy(
        update={"document_name": filename}
    )

    waiting_facts.append(extracted.model_dump())

    print(f"\nExtracted from: {filename}")
    print(extracted.model_dump())

In [0]:
assert len(waiting_facts) == 2

assert waiting_facts[0]["document_name"] == doc_02
assert waiting_facts[1]["document_name"] == doc_03

for fact in waiting_facts:
    assert isinstance(fact["initial_waiting_days"], int)
    assert isinstance(
        fact["pre_existing_disease_waiting_months"], int
    )

In [0]:
import json
import os
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ImportFormat

# Final evidence object
evidence = {
    "chat_databricks_reply": chat_databricks_reply,
    "chat_openai_reply": chat_openai_reply,
    "waiting_facts": waiting_facts
}

# Validate non-empty chat replies
assert isinstance(chat_databricks_reply, str)
assert chat_databricks_reply.strip() != ""

assert isinstance(chat_openai_reply, str)
assert chat_openai_reply.strip() != ""

w = WorkspaceClient()

output_path = (
    "/Workspace/Users/surajitm0nd0l12@gmail.com/"
    "Saffron_Shield/evidence/r3_langchain.json"
)

json_content = json.dumps(
    evidence,
    indent=4,
    ensure_ascii=False
)

# Ensure the parent folder exists (upload does not create folders)
w.workspace.mkdirs(os.path.dirname(output_path))

w.workspace.upload(
    output_path,
    json_content.encode("utf-8"),
    format=ImportFormat.AUTO,
    overwrite=True
)

print("Evidence uploaded:", output_path)

### R4 - Hybrid retriever & rerank

In [0]:
# Load the cross-encoder model
from sentence_transformers import CrossEncoder
from pyspark.sql import functions as F

reranker_model = CrossEncoder(
    "BAAI/bge-reranker-base",
    max_length=512
)

In [0]:
# Implement the custom reranker
def custom_rerank(question, candidates, top_k=5):

    if not candidates:
        return spark.createDataFrame(
            [],
            """
            chunk_id STRING,
            doc_name STRING,
            heading STRING,
            chunk_index INT,
            text STRING,
            rerank_score DOUBLE
            """
        )

    pairs = []

    for chunk in candidates:

        document_text = (
            f"Document: {chunk['doc_name']}\n"
            f"Heading: {chunk['heading']}\n"
            f"Content: {chunk['text']}"
        )

        pairs.append([question, document_text])

    scores = reranker_model.predict(
        pairs,
        batch_size=16,
        show_progress_bar=False
    )

    scored_chunks = []

    for chunk, score in zip(candidates, scores):

        scored_chunks.append({
            "chunk_id": str(chunk["chunk_id"]),
            "doc_name": str(chunk["doc_name"]),
            "heading": str(chunk["heading"]),
            "chunk_index": int(chunk["chunk_index"]),
            "text": str(chunk["text"]),
            "rerank_score": float(score)
        })

    reranked_df = spark.createDataFrame(scored_chunks)

    return (
        reranked_df
        .orderBy(F.desc("rerank_score"))
        .limit(top_k)
    )

In [0]:
# Implement hybrid retrieval
def retrieve_policy_chunks(question, top_k=5):

    # Step 1: Hybrid retrieval
    results = index.similarity_search(
        query_text=question,
        columns=[
            "chunk_id",
            "doc_name",
            "heading",
            "chunk_index",
            "text"
        ],
        num_results=20,
        query_type="hybrid",
        debug_level=1
    )

    # Step 2: Extract candidates
    candidates = results["result"]["data_array"]

    columns = [
        "chunk_id",
        "doc_name",
        "heading",
        "chunk_index",
        "text"
    ]

    candidates = [
        dict(zip(columns, row))
        for row in candidates
    ]

    # Step 3: Custom Cross-Encoder reranking
    final_results = custom_rerank(
        question,
        candidates,
        top_k=top_k
    )

    return final_results

In [0]:
test_question = (
    "What are the initial and pre-existing disease waiting periods "
    "for individual health insurance plans?"
)

test_results = retrieve_policy_chunks(test_question, top_k=5)

display(test_results)

In [0]:
# Complete question dictionary
questions = {
    "G04": "How long is the initial waiting period on a Silver or Gold individual plan?",

    "G05": "What is the pre-existing disease waiting period for an individual plan?",

    "G07": "Is there an initial waiting period on a corporate group plan bought by an employer?",

    "G08": "What is the pre-existing disease waiting period for an employee on a corporate group plan?"
}

In [0]:
# Expected first-ranked chunks
expected_first_chunks = {
    "G04": "02_Waiting_Periods_Individual_Plans.md#initial-waiting-period",

    "G05": "02_Waiting_Periods_Individual_Plans.md#pre-existing-disease-waiting-period",

    "G07": "03_Waiting_Periods_Group_Plans.md#initial-waiting-period",

    "G08": "03_Waiting_Periods_Group_Plans.md#pre-existing-disease-waiting-period"
}

In [0]:
# Run retrieval and reranking for all four questions
all_retrieval_results = {}

for question_id, question in questions.items():

    print(f"\n{'=' * 70}")
    print(f"QUESTION: {question_id}")
    print(question)
    print(f"{'=' * 70}")

    results = retrieve_policy_chunks(
        question,
        top_k=5
    )

    all_retrieval_results[question_id] = results

    display(results)

In [0]:
validation_results = []

for question_id, question in questions.items():

    results = all_retrieval_results[question_id]

    first_row = results.select("chunk_id").first()

    actual_chunk = (
        first_row["chunk_id"]
        if first_row is not None
        else None
    )

    expected_chunk = expected_first_chunks[question_id]

    validation_results.append({
        "question_id": question_id,
        "expected_chunk": expected_chunk,
        "actual_chunk": actual_chunk,
        "is_correct": actual_chunk == expected_chunk
    })

validation_df = spark.createDataFrame(validation_results)

display(validation_df)

### R5 - Grounded RAG chain

In [0]:
# Define a structured answer schema
from pydantic import BaseModel, Field

class PolicyAnswer(BaseModel):
    supported: bool = Field(
        description="True only if the retrieved policy text directly supports the answer."
    )

    answer: str = Field(
        description="A concise answer based exclusively on the retrieved policy text. "
                    "If unsupported, return exactly: not in policy"
    )

In [0]:
# Create the grounded answer chain
from langchain_core.prompts import ChatPromptTemplate

# Reuse the existing ChatDatabricks instance (chat_databricks) defined earlier
answer_llm = chat_databricks.with_structured_output(
    PolicyAnswer,
    method="function_calling"
)

answer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a policy assistant for insurance adjusters.

        Answer the user's question using ONLY the supplied retrieved policy sections.

        STRICT RULES:

        1. Use only facts explicitly stated in the retrieved policy text.
        2. Never use your general knowledge or outside information.
        3. Do not invent policy terms, benefits, premiums, exclusions,
           waiting periods, or coverage details.
        4. If the retrieved text does not explicitly support an answer,
           set supported to false and answer to exactly:
           not in policy
        5. If the answer is supported, set supported to true and provide
           a concise answer.
        6. Do not use the phrase "not in policy" in an answer that is supported.
        7. If the question asks about something unrelated to the policy
           documents, return not in policy.
        8. A lack of information about a benefit does not mean that the
           benefit is covered or excluded. Return not in policy.

        Return the structured output only.
        """
    ),
    (
        "human",
        """
        User question:
        {question}

        Retrieved policy sections:
        {context}

        Answer the question using only these sections.
        """
    )
])

answer_chain = answer_prompt | answer_llm

In [0]:
# Implement the final answer_policy_question() function
def answer_policy_question(question):

    # --------------------------------------------------
    # STEP 1: Retrieve and rerank the top 5 chunks
    # --------------------------------------------------

    retrieved_df = retrieve_policy_chunks(
        question,
        top_k=5
    )

    retrieved_chunks = retrieved_df.collect()

    if not retrieved_chunks:
        return {
            "answer": "not in policy",
            "cited_chunk_ids": []
        }

    # --------------------------------------------------
    # STEP 2: Prepare context and citation IDs
    # --------------------------------------------------

    context_parts = []
    cited_chunk_ids = []

    for rank, chunk in enumerate(retrieved_chunks, start=1):

        chunk_id = str(chunk["chunk_id"])

        cited_chunk_ids.append(chunk_id)

        context_parts.append(
            f"""
            SECTION RANK: {rank}

            CHUNK ID: {chunk_id}

            DOCUMENT: {chunk['doc_name']}

            HEADING: {chunk['heading']}

            POLICY TEXT:
            {chunk['text']}
            """
        )

    context = "\n\n".join(context_parts)

    # --------------------------------------------------
    # STEP 3: Generate a grounded answer
    # --------------------------------------------------

    response = answer_chain.invoke({
        "question": question,
        "context": context
    })

    # --------------------------------------------------
    # STEP 4: Enforce the abstention response
    # --------------------------------------------------

    if not response.supported:
        answer = "not in policy"

    else:
        answer = response.answer.strip()

        # Do not allow an unsupported phrase in a
        # supposedly supported answer.
        if not answer or "not in policy" in answer.lower():
            answer = "not in policy"

    # --------------------------------------------------
    # STEP 5: Return the required dictionary
    # --------------------------------------------------

    return {
        "answer": answer,
        "cited_chunk_ids": cited_chunk_ids[:5]
    }

In [0]:
# Test the two questions that cannot be answered
negative_questions = [
    "What is the premium for adding a pet dog?",
    "Is a gym membership covered under this health insurance policy?"
]

for question in negative_questions:

    result = answer_policy_question(question)

    print("\nQuestion:", question)
    print("Answer:", result["answer"])
    print("Cited chunks:", result["cited_chunk_ids"])

In [0]:
# Read questions from the JSON file
import json

GOLD_SET_PATH = (
    "/Volumes/claims/raw/landing/genai/gold_set.json"
)

# Read JSON from Unity Catalog Volume
gold_set_content = dbutils.fs.head(
    GOLD_SET_PATH,
    1024 * 1024
)

gold_set = json.loads(gold_set_content)

# Extract questions
gold_questions = gold_set["questions"]

print("Total questions:", len(gold_questions))

In [0]:
# Run the R5 function for all gold questions
gold_results = {}

for item in gold_questions:

    question_id = item["id"]
    question = item["question"]

    # G19 requires structured data + policy retrieval
    if question_id == "G19":
        continue

    result = answer_policy_question(question)

    gold_results[question_id] = {
        "question": question,
        "answer": result["answer"],
        "cited_chunk_ids": result["cited_chunk_ids"]
    }

    print(f"\n========== {question_id} ==========")
    print("Question:", question)
    print("Answer:", result["answer"])
    print("Citations:", result["cited_chunk_ids"])

In [0]:
# Add the two negative test questions
negative_questions = {
    "PET_DOG": "What is the premium for adding a pet dog?",
    "GYM": "Is a gym membership covered under this health insurance policy?"
}

for question_id, question in negative_questions.items():

    result = answer_policy_question(question)

    gold_results[question_id] = {
        "question": question,
        "answer": result["answer"],
        "cited_chunk_ids": result["cited_chunk_ids"]
    }

    print(f"\n========== {question_id} ==========")
    print("Question:", question)
    print("Answer:", result["answer"])
    print("Citations:", result["cited_chunk_ids"])

In [0]:
for question_id, result in gold_results.items():

    assert isinstance(result["answer"], str)

    assert isinstance(result["cited_chunk_ids"], list)

    assert 1 <= len(result["cited_chunk_ids"]) <= 5

    assert all(
        "#" in chunk_id
        and not chunk_id.startswith("/")
        for chunk_id in result["cited_chunk_ids"]
    )

print("All R5 test outputs passed structural validation.")

In [0]:
%sql
-- Create the table
CREATE TABLE IF NOT EXISTS claims.genai.gold_answers (
    question_id STRING NOT NULL,
    question STRING,
    answer STRING,
    cited_chunk_ids ARRAY<STRING>
)
USING DELTA;

In [0]:
# Create a special function for G19
import re
from langchain_core.prompts import ChatPromptTemplate

def answer_multi_hop_question(question):

    # Extract claim ID from the question
    match = re.search(r"CLM-\d+", question)

    if not match:
        raise ValueError("Claim ID not found in question.")

    claim_id = match.group(0)

    # -----------------------------------------
    # STEP 1: Retrieve structured claim facts
    # -----------------------------------------

    claim_df = spark.sql(f"""
        SELECT
            c.claim_id,
            c.policy_id,
            c.admission_date,
            c.room_rent_per_day,
            p.plan_code,
            p.effective_from
        FROM claims.silver.claims_clean c
        LEFT JOIN claims.silver.policy_scd2 p
            ON c.policy_id = p.policy_id
           AND CAST(c.admission_date AS TIMESTAMP) >= p.__START_AT
           AND (
                p.__END_AT IS NULL
                OR CAST(c.admission_date AS TIMESTAMP) < p.__END_AT
           )
        WHERE c.claim_id = '{claim_id}'
    """)

    claim_rows = claim_df.collect()

    if len(claim_rows) != 1:
        raise ValueError(
            f"Expected one claim and one matching policy version; "
            f"found {len(claim_rows)} rows."
        )

    claim_facts = claim_rows[0].asDict()

    if claim_facts["plan_code"] is None:
        raise ValueError("No policy version found for admission date.")

    # -----------------------------------------
    # STEP 2: Retrieve policy chunks
    # -----------------------------------------

    retrieved_df = retrieve_policy_chunks(
        question,
        top_k=5
    )

    retrieved_chunks = retrieved_df.collect()

    if not retrieved_chunks:
        return {
            "answer": "not in policy",
            "cited_chunk_ids": []
        }

    policy_context = "\n\n".join([
        f"""
        Chunk ID: {row['chunk_id']}
        Document: {row['doc_name']}
        Heading: {row['heading']}
        Policy text:
        {row['text']}
        """
        for row in retrieved_chunks
    ])

    citations = [
        row["chunk_id"]
        for row in retrieved_chunks
    ]

    # -----------------------------------------
    # STEP 3: Generate answer using both sources
    # -----------------------------------------

    g19_prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            """
            You are a claims assistant for insurance adjusters.
            Answer using ONLY the supplied structured claim facts and the
            retrieved policy sections. Do not invent facts. If the sources
            do not support an answer, say so explicitly.
            """
        ),
        (
            "human",
            """
            Question:
            {question}

            Structured claim facts:
            {claim_facts}

            Retrieved policy sections:
            {policy_context}

            Answer the question using both sources.
            """
        )
    ])

    g19_chain = g19_prompt | chat_databricks

    response = g19_chain.invoke({
        "question": question,
        "claim_facts": str(claim_facts),
        "policy_context": policy_context
    })

    return {
        "answer": response.content,
        "cited_chunk_ids": citations
    }

In [0]:
# Read questions from the JSON file
import json

GOLD_SET_PATH = (
    "/Volumes/claims/raw/landing/genai/gold_set.json"
)

gold_set = json.loads(
    dbutils.fs.head(GOLD_SET_PATH, 1024 * 1024)
)

gold_questions = gold_set["questions"]

user_questions = [
    {
        "id": "U01",
        "question": "What is the premium for adding a pet dog?"
    },
    {
        "id": "U02",
        "question": "Is a gym membership covered under this health insurance policy?"
    }
]

all_questions = gold_questions + user_questions

results = []

for item in all_questions:

    question_id = item["id"]
    question = item["question"]

    print(f"\nProcessing {question_id}: {question}")

    # G19 uses structured data + policy documents
    if question_id == "G19":

        result = answer_multi_hop_question(question)

    # All other questions use policy-only RAG
    else:

        result = answer_policy_question(question)

    results.append({
        "question_id": question_id,
        "question": question,
        "answer": result["answer"],
        "cited_chunk_ids": result["cited_chunk_ids"]
    })

    print("Answer:", result["answer"])
    print("Citations:", result["cited_chunk_ids"])

In [0]:
# Insert the results into the Delta table
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    ArrayType
)

schema = StructType([
    StructField("question_id", StringType(), False),
    StructField("question", StringType(), True),
    StructField("answer", StringType(), True),
    StructField(
        "cited_chunk_ids",
        ArrayType(StringType()),
        True
    )
])

answers_df = spark.createDataFrame(results, schema)

# Validate before writing
assert answers_df.count() == 21
assert answers_df.select("question_id").distinct().count() == 21

(
    answers_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("claims.genai.gold_answers")
)

In [0]:
%sql
SELECT *
FROM claims.genai.gold_answers

### R6 — Conversation memory

In [0]:
# Prepare the LangChain and MLflow components
import mlflow
import json
import os

mlflow.set_experiment("/Users/surajitm0nd0l12@gmail.com/Saffron_Shield/R6")

mlflow.langchain.autolog(log_traces=True)

In [0]:
# Create the question-rewriting chain
from langchain_core.prompts import ChatPromptTemplate

rewrite_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You rewrite follow-up questions into standalone questions.

        Use the conversation summary to understand references such as:
        "that count", "it", "this plan", and "for group plans".

        Rules:
        1. Preserve the user's original intent.
        2. Resolve references using the conversation summary.
        3. Do not answer the question.
        4. Do not introduce unsupported facts.
        5. If the question is already standalone, retain its meaning.
        6. Return only the rewritten question.
        """
    ),
    (
        "human",
        """
        Conversation summary:
        {summary}

        Latest user question:
        {question}

        Standalone question:
        """
    )
])

rewrite_chain = rewrite_prompt | chat_databricks

In [0]:
# Create the running-summary chain
summary_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        Maintain a concise running summary of an insurance policy
        conversation.

        Rules:
        - Preserve the main policy topic.
        - Preserve relevant plan names and waiting-period details.
        - Preserve important clarifications from previous turns.
        - Include the latest question and answer.
        - Do not include the complete conversation transcript.
        - Keep the summary under 100 words.
        - Do not invent facts.
        """
    ),
    (
        "human",
        """
        Previous summary:
        {previous_summary}

        Original user question:
        {question}

        Standalone question:
        {standalone_question}

        Assistant answer:
        {answer}

        Updated running summary:
        """
    )
])

summary_chain = summary_prompt | chat_databricks

In [0]:
# Create the conversation-memory function
def process_conversation_turn(
    turn_number,
    question,
    conversation_summary
):

    # -----------------------------------------
    # STEP 1: Rewrite the question
    # -----------------------------------------

    if not conversation_summary.strip():

        standalone_question = question

    else:

        rewrite_response = rewrite_chain.invoke({
            "summary": conversation_summary,
            "question": question
        })

        standalone_question = rewrite_response.content.strip()

    print("Original question:", question)
    print("Standalone question:", standalone_question)

    # -----------------------------------------
    # STEP 2: Hybrid retrieval + reranking
    # -----------------------------------------

    retrieved_df = retrieve_policy_chunks(
        standalone_question,
        top_k=5
    )

    retrieved_chunks = retrieved_df.collect()

    if not retrieved_chunks:
        return {
            "turn": turn_number,
            "question": question,
            "standalone_question": standalone_question,
            "answer": "not in policy",
            "cited_chunk_ids": [],
            "updated_summary": conversation_summary
        }

    # -----------------------------------------
    # STEP 3: Prepare retrieved context
    # -----------------------------------------

    policy_context = "\n\n".join([
        f"""
        Chunk ID: {row['chunk_id']}
        Document: {row['doc_name']}
        Heading: {row['heading']}
        Policy text:
        {row['text']}
        """
        for row in retrieved_chunks
    ])

    cited_chunk_ids = [
        row["chunk_id"]
        for row in retrieved_chunks
    ]

    # -----------------------------------------
    # STEP 4: Generate grounded answer
    # -----------------------------------------

    answer_response = answer_chain.invoke({
        "question": standalone_question,
        "context": policy_context
    })

    if answer_response.supported:
        answer = answer_response.answer.strip()

        if not answer or "not in policy" in answer.lower():
            answer = "not in policy"
    else:
        answer = "not in policy"

    # -----------------------------------------
    # STEP 5: Update running summary
    # -----------------------------------------

    summary_response = summary_chain.invoke({
        "previous_summary": conversation_summary,
        "question": question,
        "standalone_question": standalone_question,
        "answer": answer
    })

    updated_summary = summary_response.content.strip()

    # -----------------------------------------
    # STEP 6: Return turn information
    # -----------------------------------------

    return {
        "turn": turn_number,
        "question": question,
        "standalone_question": standalone_question,
        "answer": answer,
        "cited_chunk_ids": cited_chunk_ids[:5],
        "updated_summary": updated_summary
    }

In [0]:
# Run the exact three-turn scripted conversation
scripted_questions = [
    "What is the pre-existing disease waiting period on an individual plan?",
    "Does moving from Silver to Gold restart that count?",
    "And for group plans?"
]

conversation_summary = ""
conversation_turns = []

In [0]:
for turn_number, question in enumerate(scripted_questions, start=1):

    print(f"\n{'=' * 70}")
    print(f"TURN {turn_number}")
    print(f"{'=' * 70}")

    turn_result = process_conversation_turn(
        turn_number=turn_number,
        question=question,
        conversation_summary=conversation_summary
    )

    conversation_turns.append({
        "turn": turn_result["turn"],
        "question": turn_result["question"],
        "standalone_question": turn_result["standalone_question"],
        "answer": turn_result["answer"],
        "cited_chunk_ids": turn_result["cited_chunk_ids"]
    })

    # Carry only the updated summary into the next turn
    # (guard against a None/empty summary from the LLM)
    updated_summary = turn_result["updated_summary"]
    conversation_summary = updated_summary if updated_summary else ""

    print("Answer:", turn_result["answer"])
    print("Citations:", turn_result["cited_chunk_ids"])
    print("Updated summary:", conversation_summary or "(empty)")

### R7 - Evaluation + Observibility

In [0]:
# Load the gold set and identify eligible questions
import json

GOLD_SET_PATH = (
    "/Volumes/claims/raw/landing/genai/gold_set.json"
)

gold_set = json.loads(
    dbutils.fs.head(GOLD_SET_PATH, 1024 * 1024)
)

single_doc_questions = [
    item
    for item in gold_set["questions"]
    if isinstance(item.get("expected_chunks"), list)
    and len(item["expected_chunks"]) > 0
]

print("Single-document questions:", len(single_doc_questions))
print("Excluded multi-hop questions:", [
    item["id"]
    for item in gold_set["questions"]
    if not isinstance(item.get("expected_chunks"), list)
])

In [0]:
# Run retrieval and calculate hit-rate@5 and MRR
import numpy as np

evaluation_rows = []

for item in single_doc_questions:

    question_id = item["id"]
    question = item["question"]
    expected_chunks = set(item["expected_chunks"])

    # Hybrid retrieval + custom CrossEncoder reranking
    retrieved_df = retrieve_policy_chunks(
        question,
        top_k=5
    )

    retrieved_rows = retrieved_df.collect()

    retrieved_ids = [
        row["chunk_id"]
        for row in retrieved_rows
    ]

    # Find the first relevant rank
    reciprocal_rank = 0.0

    for rank, chunk_id in enumerate(retrieved_ids, start=1):
        if chunk_id in expected_chunks:
            reciprocal_rank = 1.0 / rank
            break

    hit = int(reciprocal_rank > 0)

    evaluation_rows.append({
        "question_id": question_id,
        "question": question,
        "expected_chunks": list(expected_chunks),
        "retrieved_chunk_ids": retrieved_ids,
        "hit_at_5": hit,
        "reciprocal_rank": reciprocal_rank
    })

n_questions = len(evaluation_rows)

hit_rate_at_5 = (
    sum(row["hit_at_5"] for row in evaluation_rows)
    / n_questions
)

mrr = (
    sum(row["reciprocal_rank"] for row in evaluation_rows)
    / n_questions
)

print("Number of questions:", n_questions)
print("Hit-rate@5:", round(hit_rate_at_5, 4))
print("MRR:", round(mrr, 4))

print("Hit-rate threshold met:", hit_rate_at_5 >= 0.80)
print("MRR threshold met:", mrr >= 0.70)

In [0]:
# Inspect the weakest retrieval cases
evaluation_df = spark.createDataFrame(evaluation_rows)

display(
    evaluation_df
    .orderBy("hit_at_5", "reciprocal_rank")
)

In [0]:
for row in evaluation_rows:

    if row["hit_at_5"] == 0 or row["reciprocal_rank"] < 1:

        print("\nQuestion:", row["question_id"])
        print("Expected:", row["expected_chunks"])
        print("Retrieved:", row["retrieved_chunk_ids"])
        print("Hit@5:", row["hit_at_5"])
        print("Reciprocal rank:", row["reciprocal_rank"])

In [0]:
# Create an MLflow experiment and trace the retrieval runs
import mlflow

EXPERIMENT_PATH = (
    "/Users/surajitm0nd0l12@gmail.com/Saffron_Shield/R7_Evaluation"
)

mlflow.set_experiment(EXPERIMENT_PATH)

experiment = mlflow.get_experiment_by_name(EXPERIMENT_PATH)

experiment_id = experiment.experiment_id

print("Experiment ID:", experiment_id)

In [0]:
@mlflow.trace(name="R7_Hybrid_Retrieval_Evaluation")
def traced_retrieval(question):

    retrieved_df = retrieve_policy_chunks(
        question,
        top_k=5
    )

    rows = retrieved_df.collect()

    return [
        {
            "chunk_id": row["chunk_id"],
            "doc_name": row["doc_name"],
            "heading": row["heading"],
            "rerank_score": float(row["rerank_score"])
        }
        for row in rows
    ]

In [0]:
for item in single_doc_questions:

    traced_retrieval(item["question"])

print("R7 retrieval traces generated.")

In [0]:
traces = mlflow.search_traces(
    experiment_ids=[experiment_id],
    max_results=100
)

display(traces)

In [0]:
# Prepare model answers for automated evaluation
judge_data = []

for item in single_doc_questions:

    question = item["question"]

    retrieved_df = retrieve_policy_chunks(
        question,
        top_k=5
    )

    retrieved_rows = retrieved_df.collect()

    context = "\n\n".join([
        f"Chunk ID: {row['chunk_id']}\n"
        f"Document: {row['doc_name']}\n"
        f"Heading: {row['heading']}\n"
        f"Policy text: {row['text']}"
        for row in retrieved_rows
    ])

    answer_response = answer_chain.invoke({
        "question": question,
        "context": context
    })

    answer = (
        answer_response.answer
        if answer_response.supported
        else "not in policy"
    )

    judge_data.append({
        "inputs": {
            "question": question
        },
        "outputs": {
            "response": answer,
            "retrieved_context": context
        },
        "expectations": {
            "expected_facts": item["expected_facts"]
        }
    })

print("Prepared judge records:", len(judge_data))

In [0]:
# Run the three MLflow judges
judges_status = "failed: evaluation not yet run"
eval_run_id = None
judge_scores = {}

try:

    from mlflow.genai.scorers import (
        RetrievalGroundedness,
        RelevanceToQuery,
        Correctness
    )

    scorers = [
        RetrievalGroundedness(),
        RelevanceToQuery(),
        Correctness()
    ]

    eval_result = mlflow.genai.evaluate(
        data=judge_data,
        scorers=scorers
    )

    judges_status = "ran"

    print("Evaluation completed.")
    print(eval_result)

except Exception as e:

    judges_status = f"failed: {type(e).__name__}: {str(e)}"

    print("Judge evaluation failed.")
    print(judges_status)

In [0]:
print("Evaluation result type:", type(eval_result))
print("Evaluation result attributes:", dir(eval_result))

In [0]:
# Extract the judge scores and evaluation run ID
if judges_status == "ran":

    print("Evaluation metrics:")
    print(eval_result.metrics)

    print("Evaluation run ID:")
    print(getattr(eval_result, "run_id", None))

In [0]:
if judges_status == "ran":

    eval_run_id = getattr(eval_result, "run_id", None)

    metrics = eval_result.metrics

    judge_scores = {
        "retrieval_groundedness/mean":
            metrics.get("retrieval_groundedness/mean"),

        "relevance_to_query/mean":
            metrics.get("relevance_to_query/mean"),

        "correctness/mean":
            metrics.get("correctness/mean")
    }

    judge_scores = {
        key: value
        for key, value in judge_scores.items()
        if value is not None
    }

    print(judge_scores)

In [0]:
# Create evidence/r7_eval.json
r7_evidence = {
    "hit_rate_at_5": float(hit_rate_at_5),
    "mrr": float(mrr),
    "n_questions": int(n_questions),
    "mlflow_experiment_id": str(experiment_id),
    "eval_run_id": (
        str(eval_run_id)
        if eval_run_id is not None
        else None
    ),
    "judges_status": judges_status,
    "judges": judge_scores if judges_status == "ran" else {}
}

In [0]:
assert n_questions == len(single_doc_questions)
assert r7_evidence["mlflow_experiment_id"]
assert r7_evidence["judges_status"]

if r7_evidence["judges_status"] == "failed":
    assert r7_evidence["eval_run_id"] is None
    assert r7_evidence["judges"] == {}

import json
import os
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ImportFormat

output_path = (
    "/Workspace/Users/surajitm0nd0l12@gmail.com/"
    "Saffron_Shield/Saffron_Shield_dabs/"
    "evidence/r7_eval.json"
)

w = WorkspaceClient()

# The SDK does not auto-create parent folders
w.workspace.mkdirs(os.path.dirname(output_path))

w.workspace.upload(
    output_path,
    json.dumps(
        r7_evidence,
        indent=4,
        ensure_ascii=False,
        default=float
    ).encode("utf-8"),
    format=ImportFormat.AUTO,
    overwrite=True
)

print("Saved:", output_path)

In [0]:
hit_gap = hit_rate_at_5 - 0.80
mrr_gap = mrr - 0.70

if hit_gap < mrr_gap:
    weakest_metric = "Hit-rate@5"
elif mrr_gap < hit_gap:
    weakest_metric = "MRR"
else:
    weakest_metric = "Both metrics are equally far from their thresholds"

print("Weakest metric:", weakest_metric)

## Weakest metric

The weakest metric is **Hit-rate@5**. The measured
hit-rate@5 is **1.00**, against the required threshold
of **0.80**, while MRR is **0.9722**, against the
required threshold of **0.70**.

Both retrieval metrics exceed their required thresholds,
indicating that the retrieval system performs well on
the evaluation gold set. Hit-rate@5 has the smaller
margin above its threshold, making it the weakest metric
relative to the required performance targets.

The evaluation also shows a relevance score of **1.00**
and a correctness score of **0.9444**.

No significant retrieval bottleneck is established by
these aggregate metrics alone. The next step should be
to inspect the individual questions with lower reciprocal
ranks and analyze whether hybrid candidate retrieval or
CrossEncoder reranking can further improve ranking quality.
The same gold evaluation set should then be used to
measure the improvement.

### R8 - ai_query batch adjudication

In [0]:
%sql
-- Create the probe claims table
CREATE OR REPLACE TABLE claims.genai.adjudication_probes
USING DELTA
AS
SELECT
    CAST(claim_id AS STRING) AS claim_id
FROM read_files(
    '/Volumes/claims/raw/landing/genai/adjudication_probe_claims.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

In [0]:
%sql
-- Create the adjudication verifier table claims.genai.adjudication_verifier
CREATE OR REPLACE TABLE claims.genai.adjudication_verifier
USING DELTA
AS

WITH probe_claims AS (
    SELECT DISTINCT claim_id
    FROM claims.genai.adjudication_probes
),

-- Get the claim details
claim_data AS (
    SELECT
        c.claim_id,
        c.policy_id,
        c.member_id,
        c.hospital_id,
        c.admission_date,
        c.is_pre_existing,
        c.room_rent_per_day
    FROM claims.silver.claims_clean c
    INNER JOIN probe_claims p
        ON c.claim_id = p.claim_id
),

-- Find the policy version active on admission date
policy_data AS (
    SELECT
        c.*,
        ps.plan_code,
        ps.effective_from,
        ps.__START_AT,
        ps.__END_AT
    FROM claim_data c
    LEFT JOIN claims.silver.policy_scd2 ps
        ON c.policy_id = ps.policy_id
        AND CAST(c.admission_date AS TIMESTAMP) >= ps.__START_AT
        AND (
            ps.__END_AT IS NULL
            OR CAST(c.admission_date AS TIMESTAMP) < ps.__END_AT
        )
),

-- Get plan terms
plan_data AS (
    SELECT
        p.*,
        pl.initial_waiting_days,
        pl.ped_waiting_months,
        pl.room_rent_cap_per_day,
        pl.copay_pct
    FROM policy_data p
    LEFT JOIN claims.raw.plans pl
        ON p.plan_code = pl.plan_code
),

-- Determine network status at admission
network_data AS (
    SELECT
        p.*,
        CASE
            WHEN EXISTS (
                SELECT 1
                FROM claims.raw.hospital_network_history h
                WHERE h.hospital_id = p.hospital_id
                  AND p.admission_date >= h.network_from
                  AND (
                      h.network_to IS NULL
                      OR p.admission_date < h.network_to
                  )
            )
            THEN TRUE
            ELSE FALSE
        END AS network_at_admission
    FROM plan_data p
),

-- Calculate waiting-period eligibility
waiting_checks AS (
    SELECT
        *,
        CASE
            WHEN effective_from IS NOT NULL
             AND initial_waiting_days IS NOT NULL
             AND admission_date >= DATE_ADD(
                    effective_from,
                    initial_waiting_days
                 )
            THEN TRUE
            ELSE FALSE
        END AS initial_waiting_ok,

        CASE
            WHEN COALESCE(is_pre_existing, FALSE) = FALSE
            THEN TRUE

            WHEN effective_from IS NOT NULL
             AND ped_waiting_months IS NOT NULL
             AND admission_date >= ADD_MONTHS(
                    effective_from,
                    ped_waiting_months
                 )
            THEN TRUE

            ELSE FALSE
        END AS ped_waiting_ok
    FROM network_data
),

-- Calculate final verifier fields
final_decisions AS (
    SELECT
        claim_id,

        (
            initial_waiting_ok
            AND ped_waiting_ok
        ) AS waiting_period_ok,

        network_at_admission,

        CASE
            WHEN room_rent_per_day > room_rent_cap_per_day
            THEN ROUND(
                (
                    room_rent_per_day - room_rent_cap_per_day
                ) / CAST(room_rent_per_day AS DOUBLE) * 100,
                2
            )
            ELSE 0.00
        END AS room_rent_deduction_pct,

        CAST(COALESCE(copay_pct, 0) AS INT) AS copay_pct,

        CASE
            WHEN initial_waiting_ok = FALSE
                THEN 'INITIAL WAITING'

            WHEN ped_waiting_ok = FALSE
                THEN 'PED_WAITING'

            WHEN room_rent_per_day > room_rent_cap_per_day
                THEN 'PAYABLE_WITH_DEDUCTION'

            ELSE 'PAYABLE_IN_FULL'
        END AS reason_code

    FROM waiting_checks
)

SELECT
    claim_id,

    CASE
        WHEN waiting_period_ok = TRUE THEN TRUE
        ELSE FALSE
    END AS payable,

    waiting_period_ok,
    network_at_admission,

    CAST(room_rent_deduction_pct AS DOUBLE)
        AS room_rent_deduction_pct,

    copay_pct,
    reason_code

FROM final_decisions;

In [0]:
%sql
SELECT *
FROM claims.genai.adjudication_verifier
ORDER BY claim_id;

In [0]:
%sql
SELECT
    (SELECT COUNT(*)
     FROM claims.genai.adjudication_probes) AS probe_count,

    COUNT(*) AS verifier_count,

    COUNT(DISTINCT claim_id) AS distinct_claim_count
FROM claims.genai.adjudication_verifier;

In [0]:
%sql
-- Prepare the LLM input
CREATE OR REPLACE TEMP VIEW r8_llm_input AS

SELECT
    c.claim_id,
    c.policy_id,
    c.member_id,
    c.hospital_id,
    c.admission_date,
    c.discharge_date,
    c.is_pre_existing,
    c.room_rent_per_day,
    c.room_days,
    c.billed_amount,
    c.claim_type,
    c.diagnosis_code,

    p.effective_from AS policy_start_date,
    p.plan_code,

    pl.plan_family,
    pl.initial_waiting_days,
    pl.ped_waiting_months,
    pl.room_rent_cap_per_day,
    pl.copay_pct,

    EXISTS (
        SELECT 1
        FROM claims.raw.hospital_network_history h
        WHERE h.hospital_id = c.hospital_id
          AND c.admission_date >= h.network_from
          AND (
              h.network_to IS NULL
              OR c.admission_date < h.network_to
          )
    ) AS network_at_admission

FROM claims.genai.adjudication_probes probe

INNER JOIN claims.silver.claims_clean c
    ON probe.claim_id = c.claim_id

LEFT JOIN claims.silver.policy_scd2 p
    ON c.policy_id = p.policy_id
    AND CAST(c.admission_date AS TIMESTAMP) >= p.__START_AT
    AND (
        p.__END_AT IS NULL
        OR CAST(c.admission_date AS TIMESTAMP) < p.__END_AT
    )

LEFT JOIN claims.raw.plans pl
    ON p.plan_code = pl.plan_code;

In [0]:
%sql
SELECT *
FROM r8_llm_input
ORDER BY claim_id;

In [0]:
%sql
CREATE OR REPLACE TABLE claims.genai.adjudications (
    claim_id STRING,
    payable BOOLEAN,
    waiting_period_ok BOOLEAN,
    network_at_admission BOOLEAN,
    room_rent_deduction_pct DOUBLE,
    copay_pct INT,
    reason_code STRING,
    reasoning STRING
)

In [0]:
%sql
INSERT OVERWRITE TABLE claims.genai.adjudications

WITH llm_requests AS (

    SELECT
        claim_id,

        ai_query(
            'databricks-meta-llama-3-3-70b-instruct',

            CONCAT(
                'You are an independent health insurance claim adjudicator. ',
                'Evaluate each claim strictly using the supplied data. ',

                '\n\nMANDATORY DECISION RULES:\n',

                'RULE 1 - INITIAL WAITING PERIOD:\n',
                'Calculate completion date as policy_start_date ',
                'plus initial_waiting_days. ',
                'Admission on or after completion date passes. ',
                'Otherwise payable=false and reason_code=INITIAL WAITING.\n',

                'RULE 2 - PED WAITING PERIOD:\n',
                'If is_pre_existing is false, PED waiting is not applicable. ',
                'Set PED check to passed. ',
                'If is_pre_existing is true, calculate completion date ',
                'using ADD_MONTHS(policy_start_date, ped_waiting_months). ',
                'Admission before completion means payable=false ',
                'and reason_code=PED_WAITING. ',
                'Admission on or after completion passes.\n',

                'RULE 3 - WAITING PERIOD RESULT:\n',
                'waiting_period_ok must be true only when both applicable ',
                'waiting period checks pass. ',
                'If either applicable waiting period fails, payable=false. ',
                'Do not calculate a payable deduction as the final decision ',
                'when a waiting period fails.\n',

                'RULE 4 - ROOM RENT DEDUCTION:\n',
                'If room_rent_per_day is greater than ',
                'room_rent_cap_per_day, calculate EXACTLY:\n',
                '((room_rent_per_day - room_rent_cap_per_day) ',
                '/ room_rent_per_day) * 100.\n',
                'Round to two decimal places. ',
                'If room rent is less than or equal to the cap, ',
                'deduction must be zero.\n',

                'RULE 5 - FINAL REASON CODE:\n',
                'If initial waiting fails: INITIAL WAITING.\n',
                'If PED waiting fails: PED_WAITING.\n',
                'If waiting periods pass and room rent deduction is ',
                'greater than zero: PAYABLE_WITH_DEDUCTION.\n',
                'If waiting periods pass and room rent deduction is zero: ',
                'PAYABLE_IN_FULL.\n',

                'RULE 6 - COPAY:\n',
                'Return copay_pct exactly as supplied in the plan terms. ',
                'A zero copay does NOT cancel a room rent deduction.\n',

                'RULE 7 - NETWORK:\n',
                'Return network_at_admission exactly as supplied. ',
                'Do not change it based on current network status.\n',

                '\n\nMADE-UP WORKED EXAMPLE 1:\n',
                'Start date 2025-01-01, admission 2025-07-01, ',
                'PED true, PED waiting 12 months. ',
                'PED completes 2026-01-01. ',
                'Result: payable=false, reason_code=PED_WAITING.\n',

                '\n\nMADE-UP WORKED EXAMPLE 2:\n',
                'Room rent 6500, cap 6000. ',
                'Deduction = (500 / 6500) * 100 = 7.69. ',
                'If waiting periods pass: ',
                'payable=true, reason_code=PAYABLE_WITH_DEDUCTION.\n',

                '\n\nMADE-UP WORKED EXAMPLE 3:\n',
                'Room rent 5000, cap 3000. ',
                'Deduction = (2000 / 5000) * 100 = 40.00. ',
                'Copay is 0. ',
                'Result: payable=true, deduction=40.00, ',
                'reason_code=PAYABLE_WITH_DEDUCTION.\n',

                '\n\nMADE-UP WORKED EXAMPLE 4:\n',
                'Room rent 2500, cap 3000. ',
                'Deduction=0. ',
                'If waiting periods pass: ',
                'reason_code=PAYABLE_IN_FULL.\n',

                '\n\nOUTPUT REQUIREMENTS:\n',
                'Return only the JSON object matching the required schema. ',
                'Do not omit any field. ',
                'Reasoning must explain the actual dates and calculations. ',
                'Do not contradict your numeric output in your reasoning. ',
                'Do not use the worked examples as claim facts. ',
                'Do not use any external verifier decision.\n',

                '\n\nCLAIM DATA:\n',

                TO_JSON(
                    NAMED_STRUCT(
                        'claim_id', claim_id,
                        'policy_id', policy_id,
                        'member_id', member_id,
                        'hospital_id', hospital_id,
                        'admission_date', admission_date,
                        'discharge_date', discharge_date,
                        'is_pre_existing', is_pre_existing,
                        'room_rent_per_day', room_rent_per_day,
                        'room_days', room_days,
                        'billed_amount', billed_amount,
                        'claim_type', claim_type,
                        'diagnosis_code', diagnosis_code,
                        'policy_start_date', policy_start_date,
                        'plan_code', plan_code,
                        'plan_family', plan_family,
                        'initial_waiting_days', initial_waiting_days,
                        'ped_waiting_months', ped_waiting_months,
                        'room_rent_cap_per_day', room_rent_cap_per_day,
                        'copay_pct', copay_pct,
                        'network_at_admission', network_at_admission
                    )
                )
            ),

            responseFormat => '{
                "type": "json_schema",
                "json_schema": {
                    "name": "claim_adjudication",
                    "strict": true,
                    "schema": {
                        "type": "object",
                        "properties": {
                            "payable": {"type": "boolean"},
                            "waiting_period_ok": {"type": "boolean"},
                            "network_at_admission": {"type": "boolean"},
                            "room_rent_deduction_pct": {"type": "number"},
                            "copay_pct": {"type": "integer"},
                            "reason_code": {
                                "type": "string",
                                "enum": [
                                    "INITIAL WAITING",
                                    "PED_WAITING",
                                    "PAYABLE_WITH_DEDUCTION",
                                    "PAYABLE_IN_FULL"
                                ]
                            },
                            "reasoning": {"type": "string"}
                        },
                        "required": [
                            "payable",
                            "waiting_period_ok",
                            "network_at_admission",
                            "room_rent_deduction_pct",
                            "copay_pct",
                            "reason_code",
                            "reasoning"
                        ],
                        "additionalProperties": false
                    }
                }
            }',

            modelParameters => NAMED_STRUCT(
                'temperature', CAST(0.0 AS DOUBLE),
                'max_tokens', 700
            ),

            failOnError => false

        ) AS ai_response

    FROM r8_llm_input
),

parsed_results AS (
    SELECT
        claim_id,
        ai_response.result AS response_json,
        ai_response.errorMessage AS error_message
    FROM llm_requests
)

SELECT
    claim_id,
    parsed.payable,
    parsed.waiting_period_ok,
    parsed.network_at_admission,

    CAST(ROUND(parsed.room_rent_deduction_pct, 2) AS DOUBLE),

    parsed.copay_pct,
    parsed.reason_code,
    parsed.reasoning

FROM (
    SELECT
        claim_id,

        FROM_JSON(
            response_json,
            'STRUCT<
                payable:BOOLEAN,
                waiting_period_ok:BOOLEAN,
                network_at_admission:BOOLEAN,
                room_rent_deduction_pct:DOUBLE,
                copay_pct:INT,
                reason_code:STRING,
                reasoning:STRING
            >'
        ) AS parsed

    FROM parsed_results
    WHERE error_message IS NULL
      AND response_json IS NOT NULL
) results;

In [0]:
%sql
SELECT *
FROM claims.genai.adjudications
ORDER BY claim_id;

In [0]:
%sql
SELECT
    p.claim_id
FROM claims.genai.adjudication_probes p
LEFT JOIN claims.genai.adjudications a
    ON p.claim_id = a.claim_id
WHERE a.claim_id IS NULL;

In [0]:
%sql
SELECT
    v.claim_id,

    v.payable AS verifier_payable,
    a.payable AS llm_payable,

    v.reason_code AS verifier_reason,
    a.reason_code AS llm_reason,

    CASE
        WHEN v.payable = a.payable
         AND v.waiting_period_ok = a.waiting_period_ok
         AND v.network_at_admission = a.network_at_admission
         AND ABS(
             v.room_rent_deduction_pct
             - a.room_rent_deduction_pct
         ) < 0.01
         AND v.copay_pct = a.copay_pct
         AND v.reason_code = a.reason_code
        THEN 'AGREE'
        ELSE 'DISAGREE'
    END AS comparison

FROM claims.genai.adjudication_verifier v
INNER JOIN claims.genai.adjudications a
    ON v.claim_id = a.claim_id
ORDER BY v.claim_id;

In [0]:
%sql
WITH comparison AS (
    SELECT
        CASE
            WHEN v.payable = a.payable
             AND v.waiting_period_ok = a.waiting_period_ok
             AND v.network_at_admission = a.network_at_admission
             AND ABS(
                 v.room_rent_deduction_pct
                 - a.room_rent_deduction_pct
             ) < 0.01
             AND v.copay_pct = a.copay_pct
             AND v.reason_code = a.reason_code
            THEN 1
            ELSE 0
        END AS is_agree
    FROM claims.genai.adjudication_verifier v
    INNER JOIN claims.genai.adjudications a
        ON v.claim_id = a.claim_id
)

SELECT
    COUNT(*) AS total_claims,
    SUM(is_agree) AS agreements,
    COUNT(*) - SUM(is_agree) AS disagreements,
    ROUND(100.0 * AVG(is_agree), 2) AS agreement_percentage
FROM comparison;

In [0]:
%sql
SELECT
    v.claim_id,

    v.payable AS verifier_payable,
    a.payable AS llm_payable,

    v.waiting_period_ok AS verifier_waiting_ok,
    a.waiting_period_ok AS llm_waiting_ok,

    v.network_at_admission AS verifier_network,
    a.network_at_admission AS llm_network,

    v.room_rent_deduction_pct AS verifier_deduction,
    a.room_rent_deduction_pct AS llm_deduction,

    v.copay_pct AS verifier_copay,
    a.copay_pct AS llm_copay,

    v.reason_code AS verifier_reason,
    a.reason_code AS llm_reason,

    a.reasoning AS llm_reasoning

FROM claims.genai.adjudication_verifier v
JOIN claims.genai.adjudications a
    ON v.claim_id = a.claim_id

WHERE
       NOT (v.payable <=> a.payable)
    OR NOT (v.waiting_period_ok <=> a.waiting_period_ok)
    OR NOT (v.network_at_admission <=> a.network_at_admission)
    OR ABS(
        COALESCE(v.room_rent_deduction_pct, 0)
        - COALESCE(a.room_rent_deduction_pct, 0)
    ) >= 0.01
    OR NOT (v.copay_pct <=> a.copay_pct)
    OR NOT (v.reason_code <=> a.reason_code)

ORDER BY v.claim_id;

In [0]:
%sql
SELECT
    claim_id,
    policy_start_date,
    admission_date,
    is_pre_existing,
    initial_waiting_days,
    ped_waiting_months,
    room_rent_per_day,
    room_rent_cap_per_day,
    copay_pct,
    network_at_admission
FROM r8_llm_input
WHERE claim_id IN (
    'CLM-10039',
    'CLM-11072',
    'CLM-11825',
    'CLM-12402',
    'CLM-20001',
    'CLM-20002',
    'CLM-20003'
)
ORDER BY claim_id;

In [0]:
%sql
SELECT
    claim_id,
    ai_response.errorMessage AS error_message
FROM (
    SELECT
        claim_id,
        ai_query(
            'databricks-meta-llama-3-3-70b-instruct',
            'Return the JSON object {"status":"ok"}.',
            failOnError => false
        ) AS ai_response
    FROM claims.genai.adjudication_probes
    WHERE claim_id = 'CLM-10039'
);

In [0]:
%sql
WITH test AS (
    SELECT
        ai_query(
            'databricks-meta-llama-3-3-70b-instruct',
            'Return only this JSON: {"status":"ok"}.',
            failOnError => false
        ) AS ai_result
)
SELECT
    ai_result,
    ai_result.result AS response,
    ai_result.errorMessage AS error_message
FROM test;

In [0]:
%sql
SELECT
    claim_id,
    reasoning,
    payable,
    waiting_period_ok,
    network_at_admission,
    room_rent_deduction_pct,
    copay_pct,
    reason_code
FROM claims.genai.adjudications
WHERE claim_id = 'CLM-10039';

In [0]:
%sql
SELECT
    ai_result.result AS model_response,
    ai_result.errorMessage AS error_message
FROM (
    SELECT ai_query(
        'databricks-meta-llama-3-3-70b-instruct',
        'Return only this JSON: {"status":"ok"}.',
        failOnError => false
    ) AS ai_result
);